# WP5 Old vs New Dataset & Split Comparison

This notebook compares the **legacy WP5 dataset + split** (flat layout) and the **new BumpDataset-based WP5 dataset + split**.

Goals:
- Compare old vs new **train/test splits** (sizes, serial-number overlaps, which serials moved between train/test).
- Inspect **label coverage and class distributions** in old vs new train/test sets.
- Provide context for why:
  1. Fully supervised (FS) runs improved by ~9 Dice points on the new data/split.
  2. 1% labeled experiments are now clearly worse than FS on the new data/split, while they used to be comparable on the legacy data/split.

Notes:
- The **BumpDataset `type` field (e.g. '2D', '2.5D', '3D') is intentionally ignored** when building datalists.
- All entries referenced by the split config participate in train/test splits; do **not** filter by `type` when generating `datalist_*_new.json`.

This notebook focuses strictly on **data-level differences**. Training code, model, and loss/metric policies are assumed unchanged.


## Environment and Testing Notes

- This is an **analysis notebook**, not part of the training library.
- It uses existing MONAI-style datalists and label files, which are already exercised by the training/eval scripts.
- No additional automated tests are defined here; instead, you should:
  - Run cells step by step.
  - Verify that summary tables and distributions make sense for your experiments.

The notebook is written to be fast by default (sampling cases where appropriate). You can increase sample sizes later if you want full-dataset statistics.

In [ ]:
# Imports
import json
import os
import re
from collections import Counter, defaultdict
from pathlib import Path

import numpy as np
import nibabel as nib
import pandas as pd

from tqdm import tqdm

import matplotlib.pyplot as plt
%matplotlib inline


In [ ]:
# Paths to legacy (old) and new datalists
# These default to files in this repo; adjust if needed.
ROOT = Path.cwd()

OLD_TRAIN_LIST = ROOT / "datalist_train.json"
OLD_TEST_LIST = ROOT / "datalist_test.json"

NEW_TRAIN_LIST = ROOT / "datalist_train_new.json"
NEW_TEST_LIST = ROOT / "datalist_test_new.json"

print("Old train list:", OLD_TRAIN_LIST)
print("Old test  list:", OLD_TEST_LIST)
print("New train list:", NEW_TRAIN_LIST)
print("New test  list:", NEW_TEST_LIST)

assert OLD_TRAIN_LIST.exists() and OLD_TEST_LIST.exists(), "Legacy datalists not found."
assert NEW_TRAIN_LIST.exists() and NEW_TEST_LIST.exists(), "New datalists not found."


In [ ]:
# Load datalists
def load_datalist(path: Path):
    with path.open("r") as f:
        return json.load(f)

old_train = load_datalist(OLD_TRAIN_LIST)
old_test = load_datalist(OLD_TEST_LIST)
new_train = load_datalist(NEW_TRAIN_LIST)
new_test = load_datalist(NEW_TEST_LIST)

len(old_train), len(old_test), len(new_train), len(new_test)


## Helper functions: serial numbers and basic split stats

We extract serial numbers from case IDs / filenames using the same convention as the training script:

- Serial is parsed from `SN<serial>...` at the start of the ID or image filename.
- This keeps comparisons consistent between legacy and new datasets.

We then compute basic split statistics:
- Number of records and serials per split.
- Train/test disjointness within each dataset.
- Overlaps and differences between old and new splits.

In [ ]:
SERIAL_RE = re.compile(r"^SN(\d+)")

def get_serial_from_record(rec):
    """Extract serial number from a datalist record.

    Priority:
    1) rec["id"] starting with SN<serial>
    2) basename of rec["image"] starting with SN<serial>
    Returns int serial or None if not found.
    """
    idv = rec.get("id", "") or ""
    m = SERIAL_RE.match(idv)
    if m:
        return int(m.group(1))
    bn = os.path.basename(rec["image"])
    m = SERIAL_RE.match(bn)
    if m:
        return int(m.group(1))
    return None

def summarize_split(name, data):
    serials = [get_serial_from_record(r) for r in data]
    serials = [s for s in serials if s is not None]
    counts = Counter(serials)
    return {
        "split": name,
        "n_records": len(data),
        "n_serials": len(counts),
        "min_serial": min(counts) if counts else None,
        "max_serial": max(counts) if counts else None,
        "median_cases_per_serial": float(np.median(list(counts.values()))) if counts else None,
    }, counts

summary_rows = []
serial_counts = {}
for name, data in [
    ("old_train", old_train),
    ("old_test", old_test),
    ("new_train", new_train),
    ("new_test", new_test),
]:
    row, counts = summarize_split(name, data)
    summary_rows.append(row)
    serial_counts[name] = counts

pd.DataFrame(summary_rows)


In [ ]:
# Train/test disjointness within each dataset (by serial)
old_train_serials = set(serial_counts["old_train"].keys())
old_test_serials = set(serial_counts["old_test"].keys())
new_train_serials = set(serial_counts["new_train"].keys())
new_test_serials = set(serial_counts["new_test"].keys())

print("Legacy dataset:")
print("  |train serials| =", len(old_train_serials))
print("  |test  serials| =", len(old_test_serials))
print("  |train ∩ test|  =", len(old_train_serials & old_test_serials))
print()
print("New dataset:")
print("  |train serials| =", len(new_train_serials))
print("  |test  serials| =", len(new_test_serials))
print("  |train ∩ test|  =", len(new_train_serials & new_test_serials))


### Old vs new split comparison (by serial)

Below we compare which serials appear in old vs new train/test sets:
- Serials that moved from **train→test** or **test→train**.
- Serials included only in the new dataset vs only in the legacy one.

This helps identify whether the new configuration uses a **harder test set**, or a different subset of serials overall.

In [ ]:
all_old_serials = old_train_serials | old_test_serials
all_new_serials = new_train_serials | new_test_serials

print("Overall serial coverage:")
print("  |old serials| =", len(all_old_serials))
print("  |new serials| =", len(all_new_serials))
print("  |old ∩ new|   =", len(all_old_serials & all_new_serials))
print("  |old \ new|   =", len(all_old_serials - all_new_serials))
print("  |new \ old|   =", len(all_new_serials - all_old_serials))

moved_train_to_test = old_train_serials & new_test_serials
moved_test_to_train = old_test_serials & new_train_serials

print("\nSerials that moved from old train → new test:")
print("  count:", len(moved_train_to_test))
print("  example:", sorted(list(moved_train_to_test))[:20])

print("\nSerials that moved from old test → new train:")
print("  count:", len(moved_test_to_train))
print("  example:", sorted(list(moved_test_to_train))[:20])

only_old = sorted(list(all_old_serials - all_new_serials))
only_new = sorted(list(all_new_serials - all_old_serials))

print("\nSerials only in legacy dataset (not in new BumpDataset subset):")
print("  count:", len(only_old))
print("  example:", only_old[:20])

print("\nSerials only in new dataset (not in legacy datalist):")
print("  count:", len(only_new))
print("  example:", only_new[:20])


## Label-level statistics

We now compare label statistics between old and new datasets:

- **Coverage**: fraction of voxels with labels ≥ 0 (0–4 or 6). Unlabeled voxels are `-1`.
- **Foreground proportion**: fraction of voxels with labels 1–4 (foreground classes, excluding background 0 and ignore 6).
- **Class distributions**: counts and normalized frequencies for classes 0–4 and 6.

**Important semantics (from AGENTS.md):**
- Class `-1`: unlabeled (missing data) — excluded from coverage.
- Class `0`: background — **valid label**, included in coverage.
- Classes `1–4`: foreground — valid labels.
- Class `6`: ignore/unknown — excluded from loss/metrics but still a labeled voxel.

Coverage is computed as:

`coverage = np.sum(labels >= 0) / labels.size`

By default we sample up to a limited number of cases per split to keep things fast; you can increase `max_cases` later if desired.

In [ ]:
def compute_label_stats(datalist, max_cases=None, seed=42):
    rng = np.random.default_rng(seed)
    indices = np.arange(len(datalist))
    if max_cases is not None and max_cases < len(indices):
        indices = rng.choice(indices, size=max_cases, replace=False)

    per_case_rows = []
    global_class_counts = Counter()

    for idx in tqdm(indices, desc="Computing label stats", leave=False):
        rec = datalist[int(idx)]
        lbl_path = rec["label"]
        lbl_img = nib.load(lbl_path)
        lbl = np.asanyarray(lbl_img.dataobj)

        # Coverage and basic counts
        n_vox = lbl.size
        labeled_mask = lbl >= 0
        n_labeled = int(labeled_mask.sum())
        coverage = n_labeled / float(n_vox)

        n_unlabeled = int((lbl == -1).sum())
        n_ignore = int((lbl == 6).sum())
        foreground_mask = (lbl >= 1) & (lbl <= 4)
        n_foreground = int(foreground_mask.sum())

        vals, counts = np.unique(lbl[labeled_mask], return_counts=True)
        class_counts = {int(v): int(c) for v, c in zip(vals, counts)}
        for v, c in class_counts.items():
            global_class_counts[int(v)] += int(c)

        row = {
            "id": rec.get("id", ""),
            "label_path": lbl_path,
            "coverage": coverage,
            "n_voxels": n_vox,
            "n_labeled": n_labeled,
            "n_unlabeled": n_unlabeled,
            "n_ignore": n_ignore,
            "n_foreground": n_foreground,
            "foreground_fraction": n_foreground / float(n_vox),
        }

        for cls in [0, 1, 2, 3, 4, 6]:
            row[f"class_{cls}_count"] = class_counts.get(cls, 0)

        per_case_rows.append(row)

    df = pd.DataFrame(per_case_rows)
    return df, global_class_counts


In [ ]:
# Configure how many cases to sample per split for label statistics.
# Use a small number (e.g., 32–64) first to check things quickly.
MAX_CASES_PER_SPLIT = 64  # set to None to use all cases (may be slow)

old_train_df, old_train_classes = compute_label_stats(old_train, max_cases=MAX_CASES_PER_SPLIT)
old_test_df, old_test_classes = compute_label_stats(old_test, max_cases=MAX_CASES_PER_SPLIT)
new_train_df, new_train_classes = compute_label_stats(new_train, max_cases=MAX_CASES_PER_SPLIT)
new_test_df, new_test_classes = compute_label_stats(new_test, max_cases=MAX_CASES_PER_SPLIT)


In [ ]:
def summarize_label_df(df: pd.DataFrame, name: str):
    print(f"==== {name} ====")
    print("n_cases:", len(df))
    print("coverage   (mean ± std):", df["coverage"].mean(), "+-", df["coverage"].std())
    print("foreground (mean ± std):", df["foreground_fraction"].mean(), "+-", df["foreground_fraction"].std())
    print("n_unlabeled (mean ± std):", df["n_unlabeled"].mean(), "+-", df["n_unlabeled"].std())
    print("n_ignore    (mean ± std):", df["n_ignore"].mean(), "+-", df["n_ignore"].std())
    print()

summarize_label_df(old_train_df, "old_train")
summarize_label_df(old_test_df, "old_test")
summarize_label_df(new_train_df, "new_train")
summarize_label_df(new_test_df, "new_test")


In [ ]:
def normalize_class_counts(class_counts):
    total = sum(class_counts.values())
    if total == 0:
        return {k: 0.0 for k in [0, 1, 2, 3, 4, 6]}
    return {k: class_counts.get(k, 0) / float(total) for k in [0, 1, 2, 3, 4, 6]}

rows = []
for name, counts in [
    ("old_train", old_train_classes),
    ("old_test", old_test_classes),
    ("new_train", new_train_classes),
    ("new_test", new_test_classes),
]:
    freq = normalize_class_counts(counts)
    row = {"split": name}
    row.update({f"freq_class_{k}": v for k, v in freq.items()})
    rows.append(row)

class_freq_df = pd.DataFrame(rows)
class_freq_df


In [ ]:
# Quick visualization: foreground fraction distributions
fig, axs = plt.subplots(1, 2, figsize=(12, 4), sharey=True)

axs[0].hist(old_train_df["foreground_fraction"], bins=20, alpha=0.7, label="old_train")
axs[0].hist(old_test_df["foreground_fraction"], bins=20, alpha=0.7, label="old_test")
axs[0].set_title("Legacy dataset: foreground fraction")
axs[0].set_xlabel("Foreground voxels / total voxels")
axs[0].legend()

axs[1].hist(new_train_df["foreground_fraction"], bins=20, alpha=0.7, label="new_train")
axs[1].hist(new_test_df["foreground_fraction"], bins=20, alpha=0.7, label="new_test")
axs[1].set_title("New dataset: foreground fraction")
axs[1].set_xlabel("Foreground voxels / total voxels")
axs[1].legend()

plt.tight_layout()
plt.show()


## Next steps / how to interpret results

Once you run this notebook end-to-end, you should be able to answer questions like:

- Did the **new train/test split** move harder serials from train to test, or vice versa?
- Are there serials present only in the new dataset (or only in the legacy one)?
- Do **coverage** and **foreground fractions** differ systematically between old vs new train/test sets?
- Are rare classes (3, 4) more or less prevalent in the new train vs test sets compared to the legacy layout?

Those data-level differences can explain why:
- Fully supervised runs may improve on the new configuration (e.g., easier test set, better coverage, more consistent labels).
- 1% labeled experiments may now lag FS (e.g., fewer easy cases in train, lower coverage or different class balance, or harder test serials).

If you discover a specific pattern (e.g., new test set dominated by low-coverage cases, or rare-class-heavy serials moved from train to test), you can then design targeted follow-up experiments.